# Estudio 06 · Fase 1b′ · Humedad de ERA5-Land (L + 5a + H)

Preregistro `PHASE1B_PRIME` (commit `ebad7be`, aprobado por Rosa tal cual). Candidata: L + 5a + H, con
H = `hum_rel_media_media5_h2` (definición de gold.py) sobre la humedad semanal de ERA5-Land 2000–2025
(`silver/meteo_humedad_era5_land_2000_2025.csv`). Referencia: L + 5a. Regla de 5 temporadas de las fases 2 y 3;
se adopta con «gana» en >= 2 de 3 horizontes y sin «pierde» ni «compensa».

**Disponibilidad (decisión de Rosa, 2026-11-04):** la app corre cuando hay casos nuevos, y la Sala Situacional del
MINSA va unas 3 semanas atrasada (una sola observación: en la semana epi 44 de 2026 seguía en la 41). Como
referencia conservadora, un dato se considera disponible si se publica a más tardar en `origen_cierre` + 14 días
(*hipótesis*). ERA5-Land se publica unos 5 días después de cada día (documentación de Open-Meteo).


In [1]:
import json
import sys

import numpy as np
import pandas as pd

from denguekay.config.paths import GOLD, RESEARCH
from denguekay.evaluation import episodes as diag
from denguekay.evaluation import protocol as pa
from denguekay.model import probabilistic_output as sp
from denguekay.model import tracking
from denguekay.model.device import xgboost_settings

STUDY = RESEARCH / "06_model_improvement"
sys.path.insert(0, str(STUDY / "code"))
import phase0_diagnosis as p0
import phase1_history as p1
import phase1b_prime as pb
import phase2_calibration as p2
import phase3_onsets as p3
import phase5_counts as p5
import phase6_other_families as p6
import preregistration as pr

pd.set_option("display.width", 220)
pd.set_option("display.max_columns", 40)
pd.set_option("display.max_rows", 300)
METRICS = STUDY / "docs" / "metricas" / "fase1b_prima.json"
OUTPUT_DIR = p3.OUTPUT_DIR.parent / "fase1b_prima"
REF_DAYS = 14
device = xgboost_settings()
inp = p3.phase3_inputs()
weekly = pb.era5_land_panel(inp["panel"])
origins = pb.add_humidity(inp["origenes"], pb.humidity_by_origin_week(weekly))
device


{'device': 'cuda', 'n_jobs': 12}

## 1 · Guardas G17–G20 y disponibilidad (antes de mirar resultados; G18 es el test de no fuga)


In [2]:
gold = pd.read_csv(GOLD / "meteo_socio_epi_piura_2017_2025_h2_gold.csv", dtype={"ubigeo": str},
                   float_precision="round_trip")
g17 = pb.guard_g17(gold, inp["panel"])
g19 = pb.guard_g19(origins)
base = p6.l_plus_5a(inp["serie"])
saved5 = json.loads((STUDY / "docs" / "metricas" / "fase5.json").read_text(encoding="utf-8"))
g20 = p6.guard_g16_reproduction(base["evaluaciones"], saved5["veredictos"]["5a"]["metricas"])
since = pd.to_datetime(origins.origen_cierre) >= pd.Timestamp("2005-01-01")
o = origins[since]
published = pd.to_datetime(o.h_ultima_semana) + pd.Timedelta(days=6 + pb.ERA5_LAND_DELAY_DAYS)
availability = {"referencia": f"origen_cierre + {REF_DAYS} días (hipótesis)", "origenes": int(len(o)),
                "afectados": int((published > pd.to_datetime(o.origen_cierre) + pd.Timedelta(days=REF_DAYS)).sum()),
                "afectados_contra_origen_cierre": int((published > pd.to_datetime(o.origen_cierre)).sum())}
availability["pasa"] = availability["afectados"] == 0
if not (g17["pasa"] and g19["pasa"] and g20["pasa"] and availability["pasa"]):
    raise RuntimeError(f"Falla una guarda: se detiene la fase. G17={g17} G19={g19} G20={g20} disp={availability}")
{"G17": g17, "G19": g19, "G20": g20, "disponibilidad": availability}


{'G17': {'filas_gold': 30485,
  'con_valor': 30095,
  'mismos_vacios': True,
  'diferencia_maxima': 0.0,
  'pasa': True},
 'G19': {'origenes_desde_2005': 71110, 'sin_H': 0, 'pasa': True},
 'G20': {'diferencia_maxima': 4.9451664985156185e-11,
  'filas': 18,
  'pasa': True},
 'disponibilidad': {'referencia': 'origen_cierre + 14 días (hipótesis)',
  'origenes': 71110,
  'afectados': 0,
  'afectados_contra_origen_cierre': 71110,
  'pasa': True}}

## 2 · Candidata L + 5a + H (5 semillas)


In [3]:
columns, hp = p3._config()
columns = [*columns, pb.COLUMN]
blocks = p1.seasonal_blocks(p1.FIRST_BLOCK_SEASON_LONG)
preds, evals, fits = {}, {}, []
for seed in pr.SEEDS:
    pred = p3.predict_blocks_weighted(inp["matrices"], origins, blocks, seed, columns, hp)
    fits.append(pred.attrs.pop("ajustes").assign(semilla=seed))
    preds[seed] = pred
    evals[seed] = p5.evaluate_mixture(pred, inp["serie"])
cand = {"predicciones": preds, "evaluaciones": evals}
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
pd.concat(preds.values(), ignore_index=True).to_csv(OUTPUT_DIR / "predicciones_1b_prima.csv.gz", index=False)
pd.concat(fits, ignore_index=True).drop(columns=["corte"]).to_csv(OUTPUT_DIR / "ajustes_1b_prima.csv", index=False)


## 3 · Veredicto preregistrado


In [4]:
dec = p3.compare(cand, base)
v = dec["veredicto"]
won = int(v.veredicto.eq("gana").sum())
bad = bool(v.veredicto.isin(["pierde", "compensa (empate)"]).any())
adopted = won >= pr.PHASE1B_PRIME_MIN_HORIZONS_WON and not bad
print("Horizontes ganados:", won, "| pierde o compensa:", bad, "| se adopta H:", adopted)
v


Horizontes ganados: 0 | pierde o compensa: False | se adopta H: False


,horizonte,veredicto,mejora_antelacion,empeora_antelacion,vetos,vetos_contra_referencia,a_favor_f1_050,a_favor_auprc,a_favor_brier,a_favor_razon_mae,en_contra_f1_050,en_contra_auprc,en_contra_brier,en_contra_razon_mae,a_favor_antelacion,en_contra_antelacion
0,2,empate,False,False,None,None,0,0,0,1,0,1,1,0,1,0
1,3,empate,False,False,None,None,0,1,0,2,1,2,1,0,0,0
2,4,empate,False,False,None,None,0,1,0,2,1,1,1,0,0,0


In [5]:
dec["antelacion"].round(3)


,horizonte,episodios,diferencia,ic95_inferior,ic95_superior,diferencia_temporada_2010,diferencia_temporada_2015,diferencia_temporada_2016,diferencia_temporada_2022,diferencia_temporada_2023,episodios_temporada_2010,episodios_temporada_2015,episodios_temporada_2016,episodios_temporada_2022,episodios_temporada_2023
0,2,193,0.010,-0.016,0.036,0.0,-0.029,0.029,0.000,0.035,30,34,34,38,57
1,3,193,-0.010,-0.031,0.010,0.0,0.000,-0.029,-0.026,0.000,30,34,34,38,57
2,4,193,-0.005,-0.016,0.000,0.0,0.000,0.000,-0.026,0.000,30,34,34,38,57


In [6]:
cols = ["horizonte", "bloque", "positivos_cand", "f1_050_cand", "f1_050_ref", "auprc_cand", "auprc_ref",
        "brier_cand", "brier_ref", "razon_mae_cand", "razon_mae_ref", "razon_rmse_cand", "razon_rmse_ref"]
dec["metricas"][cols].round(3)


,horizonte,bloque,positivos_cand,f1_050_cand,f1_050_ref,auprc_cand,auprc_ref,brier_cand,brier_ref,razon_mae_cand,razon_mae_ref,razon_rmse_cand,razon_rmse_ref
3,2,temporada_2010,234.0,0.710,0.722,0.780,0.780,0.030,0.030,0.919,0.907,0.909,0.899
8,2,temporada_2015,557.0,0.767,0.772,0.884,0.885,0.052,0.051,0.891,0.889,0.937,0.932
9,2,temporada_2016,395.0,0.660,0.657,0.755,0.757,0.059,0.059,1.034,1.030,1.000,0.995
15,2,temporada_2022,431.0,0.710,0.712,0.779,0.781,0.054,0.054,0.942,0.946,0.944,0.965
16,2,temporada_2023,1039.0,0.859,0.857,0.948,0.947,0.062,0.062,0.793,0.808,0.864,0.893
17,2,temporada_2024,857.0,0.803,0.799,0.895,0.893,0.073,0.074,0.947,1.136,0.969,1.316
21,3,temporada_2010,234.0,0.615,0.631,0.707,0.706,0.036,0.036,0.889,0.891,0.889,0.890
26,3,temporada_2015,557.0,0.631,0.649,0.838,0.839,0.065,0.064,0.833,0.828,0.919,0.914
27,3,temporada_2016,395.0,0.627,0.627,0.707,0.709,0.065,0.065,1.047,1.042,1.021,1.016
33,3,temporada_2022,431.0,0.700,0.704,0.769,0.771,0.056,0.056,0.912,0.927,0.949,0.990


## 4 · Lecturas que no deciden


In [7]:
display(dec["lectura_sin_2010"][["horizonte", "veredicto", "vetos"]])
mae_rule = p5.compare(cand, base)
mae_rule["veredicto"][["horizonte", "veredicto", "vetos"]]


,horizonte,veredicto,vetos
0,2,empate,None
1,3,empate,None
2,4,empate,None


,horizonte,veredicto,vetos
0,2,empate,None
1,3,empate,None
2,4,empate,None


In [8]:
rd = {k: p2.readings(x, inp["serie"]) for k, x in {"L + 5a": base, "L + 5a + H": cand}.items()}
recall = pd.concat({k: x["recall_por_tipo"] for k, x in rd.items()}, names=["variante"]).reset_index(level=0)
deciding = [pr.block_name(s) for s in pr.DECIDING_SEASONS]
new = recall[recall.tipo.eq("brote_nuevo") & recall.bloque.isin(deciding)]
new_wide = new.pivot_table(index=["horizonte", "bloque"], columns="variante", values="recall")
new_wide["semanas"] = new[new.variante.eq("L + 5a")].set_index(["horizonte", "bloque"]).semanas
diff = new_wide["L + 5a + H"] - new_wide["L + 5a"]
print("Recall de brotes nuevos: mejora en", int((diff > 0).sum()), "celdas, empeora en", int((diff < 0).sum()), "de", len(diff))
new_wide.round(3)


Recall de brotes nuevos: mejora en 2 celdas, empeora en 7 de 15


variante                  L + 5a  L + 5a + H  semanas
horizonte bloque                                     
2         temporada_2010   0.118       0.078     51.0
          temporada_2015   0.067       0.050     60.0
          temporada_2016   0.132       0.132     53.0
          temporada_2022   0.122       0.102     49.0
          temporada_2023   0.174       0.186     86.0
3         temporada_2010   0.059       0.059     51.0
          temporada_2015   0.033       0.033     60.0
          temporada_2016   0.113       0.094     53.0
          temporada_2022   0.163       0.122     49.0
          temporada_2023   0.186       0.198     86.0
4         temporada_2010   0.020       0.020     51.0
          temporada_2015   0.017       0.017     60.0
          temporada_2016   0.057       0.038     53.0
          temporada_2022   0.122       0.102     49.0
          temporada_2023   0.174       0.174     86.0

In [9]:
# Frente a la V2 en 2022 y 2023 (predicciones guardadas del estudio 05; su Platt y su clima son los suyos)
v2_evals = p0.evaluate_seeds(p0.v2_predictions(), inp["serie"])
med = {k: pa.seed_median([e["metricas"] for e in x.values()]) for k, x in
       {"V2": v2_evals, "L + 5a": base["evaluaciones"], "L + 5a + H": cand["evaluaciones"]}.items()}
keep = ["horizonte", "bloque", "positivos", "f1_050", "recall_050", "auprc", "brier", "razon_mae", "razon_rmse"]
v2_table = pd.concat({k: m[m.bloque.isin(["temporada_2022", "temporada_2023"])][keep] for k, m in med.items()},
                     names=["variante"]).reset_index(level=0)
hu = pd.concat({k: sp.hu0007_4_compliance(m[m.bloque.isin(["temporada_2022", "temporada_2023"])]) for k, m in med.items()},
               names=["variante"]).reset_index(level=0)
display(v2_table.round(3))
hu


,variante,horizonte,bloque,positivos,f1_050,recall_050,auprc,brier,razon_mae,razon_rmse
2,V2,2,temporada_2022,431.0,0.709,0.687,0.782,0.054,0.919,0.938
3,V2,2,temporada_2023,1039.0,0.858,0.821,0.948,0.063,0.818,0.869
7,V2,3,temporada_2022,431.0,0.594,0.462,0.773,0.059,0.883,0.922
8,V2,3,temporada_2023,1039.0,0.828,0.768,0.930,0.075,0.747,0.818
12,V2,4,temporada_2022,431.0,0.333,0.202,0.741,0.068,0.838,0.905
13,V2,4,temporada_2023,1039.0,0.811,0.733,0.920,0.084,0.708,0.779
15,L + 5a,2,temporada_2022,431.0,0.712,0.696,0.781,0.054,0.946,0.965
16,L + 5a,2,temporada_2023,1039.0,0.857,0.828,0.947,0.062,0.808,0.893
33,L + 5a,3,temporada_2022,431.0,0.704,0.682,0.771,0.056,0.927,0.990
34,L + 5a,3,temporada_2023,1039.0,0.835,0.794,0.930,0.073,0.763,0.852


,variante,horizonte,bloque,precision_050,recall_050,f1_050,razon_mae,razon_rmse,cumple_precision_050,cumple_recall_050,cumple_f1_050,cumple_razon_mae,cumple_razon_rmse,cumple_clasificacion,cumple_regresion
2,V2,2,temporada_2022,0.734491,0.686775,0.709288,0.918743,0.937701,True,False,True,False,False,False,False
3,V2,2,temporada_2023,0.898841,0.820982,0.858149,0.818114,0.868532,True,True,True,True,False,True,False
7,V2,3,temporada_2022,0.835391,0.461717,0.594030,0.883139,0.921506,True,False,False,False,False,False,False
8,V2,3,temporada_2023,0.896745,0.768046,0.827801,0.747036,0.818020,True,False,True,True,True,False,True
12,V2,4,temporada_2022,0.959596,0.201856,0.333333,0.838328,0.905236,True,False,False,True,False,False,False
13,V2,4,temporada_2023,0.911483,0.733397,0.811129,0.707967,0.779131,True,False,True,True,True,False,True
15,L + 5a,2,temporada_2022,0.728155,0.696056,0.711905,0.945681,0.964720,True,False,True,False,False,False,False
16,L + 5a,2,temporada_2023,0.888545,0.827719,0.857428,0.807630,0.892501,True,True,True,True,False,True,False
33,L + 5a,3,temporada_2022,0.727723,0.682135,0.704327,0.927262,0.989964,True,False,True,False,False,False,False
34,L + 5a,3,temporada_2023,0.880342,0.794033,0.835020,0.762822,0.852290,True,False,True,True,False,False,False


In [10]:
# Humedad ERA5-Land − «Best match» 2017-2025 por año (65 distritos)
current = inp["panel"][["ubigeo", "semana_inicio", "anio", "hum_rel_media"]].copy()
current["semana_inicio"] = pd.to_datetime(current.semana_inicio)
current["ubigeo"] = current.ubigeo.astype(str)
both = current.merge(weekly, on=["ubigeo", "semana_inicio"], suffixes=("_best_match", "_era5_land"))
both["dif"] = both.hum_rel_media_era5_land - both.hum_rel_media_best_match
hum_diff = both.groupby("anio").agg(semanas=("dif", "size"), diferencia_media=("dif", "mean"),
                                    mae=("dif", lambda d: d.abs().mean()),
                                    correlacion=("dif", lambda d: both.loc[d.index, "hum_rel_media_era5_land"].corr(
                                        both.loc[d.index, "hum_rel_media_best_match"])))
hum_diff.round(3)


,semanas,diferencia_media,mae,correlacion
anio,,,,
2017,3380,1.017,3.011,0.869
2018,3380,0.613,3.451,0.853
2019,3380,1.485,3.752,0.860
2020,3445,0.779,3.557,0.867
2021,3380,1.553,3.876,0.846
2022,3380,1.067,3.626,0.846
2023,3380,-0.221,3.279,0.879
2024,3380,0.118,3.718,0.839
2025,3380,1.282,3.916,0.838


## 5 · Salidas: métricas y MLflow


In [11]:
to_json = lambda d: {k: diag.to_json(x) for k, x in d.items() if isinstance(x, pd.DataFrame)}
content = {
    "dispositivo": device, "guardas": {"G17": g17, "G19": g19, "G20": g20, "disponibilidad": availability},
    "veredicto": to_json(dec), "adoptada": bool(adopted), "lectura_regla_mae": to_json(mae_rule),
    "recall_brotes_nuevos": diag.to_json(new_wide.reset_index()), "frente_a_v2": diag.to_json(v2_table),
    "hu0007_4_2022_2023": diag.to_json(hu), "humedad_era5_land_menos_best_match": diag.to_json(hum_diff.reset_index()),
}
content = json.loads(json.dumps(content, default=lambda o: o.item() if hasattr(o, "item") else str(o)))
for section, value in content.items():
    diag.save_metrics_section(METRICS, section, value)
tracking.log_study(
    "mejora_modelo_fase1b_prima_humedad", pd.DataFrame(columns=["numero", "estado"]),
    parent={"params": {"fase": "1b_prima", "preregistro": "ebad7be", "adoptada": str(bool(adopted)),
                       "veredicto": "; ".join(f"h{r.horizonte}: {r.veredicto}" for r in v.itertuples()),
                       **{f"xgb.{k}": str(x) for k, x in device.items()}},
            "metricas": {}},
    artifacts=[METRICS], tables={"veredicto": v},
    description="Fase 1b′ del estudio 06: L + 5a + humedad de ERA5-Land (H, alineación h=2).")


C:\Users\Rosa\RufoEsEternoElRegreso\Denguekay\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


MLflow: estudio baa635b12e0642adbf82574ccf71e1a7 en 'denguekay/mejora_modelo_fase1b_prima_humedad' (sqlite:///C:/Users/Rosa/RufoEsEternoElRegreso/Denguekay/mlflow.db)


'baa635b12e0642adbf82574ccf71e1a7'